In [1]:
import os
import rasterio
import geopandas as gpd
from rasterio.features import shapes
import warnings
warnings.filterwarnings('ignore')

# 1. PC Paths
BASE_DIR = ".."
FLOOD_8M_PATH = os.path.join(BASE_DIR, "data", "stages", "gilgit_flood_8m.tif")
ROADS_PATH = os.path.join(BASE_DIR, "data", "processed", "gilgit_roads.gpkg")
BUILDINGS_PATH = os.path.join(BASE_DIR, "data", "processed", "gilgit_buildings.gpkg")

print("Vectorizing 8m flood raster into polygons...")

# Read the raster and extract geometries
with rasterio.open(FLOOD_8M_PATH) as src:
    flood_raster = src.read(1)
    # Only trace pixels that are flooded (value == 1)
    mask = flood_raster == 1 
    
    results = (
        {'properties': {'raster_val': v}, 'geometry': s}
        for i, (s, v) in enumerate(shapes(flood_raster, mask=mask, transform=src.transform))
    )
    
    # Build the GeoDataFrame and enforce the exact CRS of the raster (UTM Zone 43N)
    flood_gdf = gpd.GeoDataFrame.from_features(list(results), crs=src.crs)

print(f"Generated {len(flood_gdf)} discrete flood polygons.")

Vectorizing 8m flood raster into polygons...
Generated 15005 discrete flood polygons.


In [2]:
import os
import urllib.request
import zipfile
import geopandas as gpd
from shapely.geometry import box
import warnings
warnings.filterwarnings('ignore')

# 1. Base Paths
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "data")
GEOFABRIK_ZIP = os.path.join(DATA_DIR, "pakistan-latest-free.shp.zip")
SHP_DIR = os.path.join(DATA_DIR, "pakistan_shp")
out_dir = os.path.join(DATA_DIR, "vulnerability")
os.makedirs(out_dir, exist_ok=True)

print("Overpass API bypassed. Initiating bulk database extraction...")

# 2. Download Real Bulk Data
url = "https://download.geofabrik.de/asia/pakistan-latest-free.shp.zip"
if not os.path.exists(GEOFABRIK_ZIP):
    print("Downloading bulk OSM data from Geofabrik (~350MB). Please wait...")
    urllib.request.urlretrieve(url, GEOFABRIK_ZIP)
    print("Download complete.")
else:
    print("Geofabrik Zip already exists locally. Skipping download.")

# 3. Extract Archive
if not os.path.exists(SHP_DIR):
    print("Extracting shapefiles...")
    with zipfile.ZipFile(GEOFABRIK_ZIP, 'r') as zip_ref:
        zip_ref.extractall(SHP_DIR)

# 4. Create a Spatial Bounding Box from the Flood Layer
# We must project our UTM flood bounds back to Degrees (EPSG:4326) to filter the raw shapefile
bounds = flood_gdf.total_bounds # minx, miny, maxx, maxy
bbox_poly = box(*bounds)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_poly], crs=flood_gdf.crs)
bbox_4326 = bbox_gdf.to_crs("EPSG:4326")

print("Loading and clipping real infrastructure data to the Gilgit bounding box...")

# 5. Targeted Loading
# The 'bbox' parameter forces GeoPandas to only load data inside our specific rectangle, saving RAM.
roads_raw = gpd.read_file(os.path.join(SHP_DIR, "gis_osm_roads_free_1.shp"), bbox=bbox_4326)
buildings_raw = gpd.read_file(os.path.join(SHP_DIR, "gis_osm_buildings_a_free_1.shp"), bbox=bbox_4326)

# Project back to Metric (UTM) for accurate engineering calculations
real_roads = roads_raw.to_crs(flood_gdf.crs)
real_buildings = buildings_raw.to_crs(flood_gdf.crs)

print("Executing spatial intersection with true data...")

# 6. Spatial Join
print("Executing geometric clipping for true road lengths...")

# 1. Buildings: sjoin is correct here. 
# If floodwater touches a building footprint, the entire building is compromised.
flooded_buildings = gpd.sjoin(real_buildings, flood_gdf, how="inner", predicate="intersects")

# 2. Roads: THE FIX. 
# We physically cut the road LineStrings at the exact boundary of the flood polygon.
clipped_roads = gpd.clip(real_roads, flood_gdf)

print(f"=== TRUE IMPACT REPORT (8m GLOF) ===")
print(f"-> Vulnerable Buildings: {len(flooded_buildings)}")
print(f"-> Vulnerable Road Segments (Clipped): {len(clipped_roads)}")

# Save the corrected geometries to disk
clipped_roads.to_file(os.path.join(out_dir, "impacted_roads_real_clipped.gpkg"), driver="GPKG")

# 3. The Accurate Levee Earthwork Calculation
true_road_length_m = clipped_roads.geometry.length.sum()

design_flood_depth = 8.0 
freeboard = 1.0 
h = design_flood_depth + freeboard 
W = 3.0 
Z = 3.0 

# Cross-sectional Area: A = h * (W + Z * h)
cross_section_area = h * (W + Z * h)
true_earthwork_volume = cross_section_area * true_road_length_m

print("\n=== REVISED ENGINEERING SOLUTION PROPOSAL ===")
print(f"Hazard: 8-Meter Glacial Lake Outburst Flood (GLOF)")
print(f"True Asset to Protect: {true_road_length_m:,.2f} meters of critical roadway")
print(f"Levee Design Height: {h}m (including 1m freeboard)")
print(f"Required Earthwork Volume: {true_earthwork_volume:,.2f} cubic meters")

Overpass API bypassed. Initiating bulk database extraction...
Geofabrik Zip already exists locally. Skipping download.
Loading and clipping real infrastructure data to the Gilgit bounding box...
Executing spatial intersection with true data...
Executing geometric clipping for true road lengths...
=== TRUE IMPACT REPORT (8m GLOF) ===
-> Vulnerable Buildings: 4867
-> Vulnerable Road Segments (Clipped): 734

=== REVISED ENGINEERING SOLUTION PROPOSAL ===
Hazard: 8-Meter Glacial Lake Outburst Flood (GLOF)
True Asset to Protect: 182,487.37 meters of critical roadway
Levee Design Height: 9.0m (including 1m freeboard)
Required Earthwork Volume: 49,271,590.55 cubic meters
